[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balloontip/deep-learning/blob/main/chapter-09/09-05-Transfer-Learning-in-PyTorch-Step-by-Step-Oak-vs-Palm-Classifier.ipynb)

10-05-Transfer-Learning-in-PyTorch-Step-by-Step-Oak-vs-Palm-Classifier

In [1]:
# -----------------------------------------------------------
# Step 1: Import Required Libraries
# -----------------------------------------------------------
import torch
import torchvision.transforms as transforms
from torchvision import models, datasets
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import time

print("Step 1: Libraries imported successfully.")

# -----------------------------------------------------------
# Step 2: Prepare the Dataset
# -----------------------------------------------------------
# In a real project, you would have a directory structure like:
#
# data/
#   train/
#     oak_tree/
#       image1.jpg
#       image2.jpg
#     palm_tree/
#       image1.jpg
#       image2.jpg
#   test/
#     oak_tree/
#       image1.jpg
#     palm_tree/
#       image1.jpg
#
# Then you would use:
# train_dataset = datasets.ImageFolder("data/train", transform=transform)
# test_dataset  = datasets.ImageFolder("data/test", transform=transform)
#
# For this fully runnable textbook example, we use FakeData instead.

print("\nStep 2: Setting up a fake dataset and data loaders.")

# ImageNet preprocessing and normalization.
# ResNet-50 was pretrained on ImageNet, so we use the same normalization.
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.FakeData(
    size=100,
    image_size=(3, 224, 224),
    num_classes=2,
    transform=transform
)

test_dataset = datasets.FakeData(
    size=20,
    image_size=(3, 224, 224),
    num_classes=2,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

class_names = ["oak_tree", "palm_tree"]

# -----------------------------------------------------------
# Step 3: Load a Pretrained ResNet-50 Model
# -----------------------------------------------------------
print("\nStep 3: Loading the pretrained ResNet-50 model.")

weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

print("ResNet-50 model loaded successfully.")

# -----------------------------------------------------------
# Step 4: Modify the Final Layer and Freeze the Backbone
# -----------------------------------------------------------
print("\nStep 4: Replacing the final layer and freezing the backbone.")

# Freeze all pretrained layers.
for param in model.parameters():
    param.requires_grad = False

# Replace the original ImageNet classifier with a new binary classifier.
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

# The new final layer is trainable by default.
print("The final classification layer has been replaced.")

# -----------------------------------------------------------
# Step 5: Define Loss Function and Optimizer
# -----------------------------------------------------------
criterion = nn.CrossEntropyLoss()

# Only train the new final layer.
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("\nStep 5: Loss function and optimizer defined.")

# -----------------------------------------------------------
# Step 6: Train the Model
# -----------------------------------------------------------
print("\nStep 6: Starting the training process.")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 5
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"Training finished in {(end_time - start_time):.2f} seconds.")

# -----------------------------------------------------------
# Step 7: Evaluate the Model
# -----------------------------------------------------------
print("\nStep 7: Evaluating the model on the test data.")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on test set: {accuracy:.2f}%")

# -----------------------------------------------------------
# Step 8: Make a Prediction on One Sample
# -----------------------------------------------------------
print("\nStep 8: Making a prediction on one sample image.")

model.eval()

sample_image, sample_label = test_dataset[0]
sample_image = sample_image.unsqueeze(0).to(device)

with torch.no_grad():
    output = model(sample_image)
    _, predicted = torch.max(output, 1)

predicted_class = class_names[predicted.item()]
actual_class = class_names[sample_label]

print(f"Predicted class: {predicted_class}")
print(f"Actual class: {actual_class}")


Step 1: Libraries imported successfully.

Step 2: Setting up a fake dataset and data loaders.

Step 3: Loading the pretrained ResNet-50 model.
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 128MB/s]


ResNet-50 model loaded successfully.

Step 4: Replacing the final layer and freezing the backbone.
The final classification layer has been replaced.

Step 5: Loss function and optimizer defined.

Step 6: Starting the training process.
Epoch 1/5, Loss: 0.7191
Epoch 2/5, Loss: 0.6855
Epoch 3/5, Loss: 0.6471
Epoch 4/5, Loss: 0.6210
Epoch 5/5, Loss: 0.5860
Training finished in 130.60 seconds.

Step 7: Evaluating the model on the test data.
Accuracy on test set: 90.00%

Step 8: Making a prediction on one sample image.
Predicted class: oak_tree
Actual class: palm_tree
